# User Interface

This Notebook generates a simple user interface using Gradio and implements the fake news model and the bias news model to make predictions on new articles that can be cut and pasted into the user interface.

*Note - You cannot cut and paste into the notebook version of the UI.  You must cut and paste the text into the browser pop-up of the user interface.*

In [15]:
# basic impoorts
import pandas as pd
from pandas import DataFrame
import numpy as np

# machine learning imports
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn import svm
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression

#user interface
import gradio as gr
import sqlite3
import calendar

from joblib import load
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import re
import string
import regex as re

#### Load Functions from Other Notebooks

First we use joblib to load the vectorizers and models for the news bias model and the fake news model.

In [6]:
# load vectorizer and chosen model for bias detection
bias_vectorizer = load("vec_text.joblib")
bias_model = load("grid_best_estimator_logreg.joblib")

In [9]:
# load vectorizer and chosen model for real/fake news detector
real_vectorizer = load("vectorizer_rn.joblib")
real_model = load("log_reg_rn.joblib")

#### Define New Functions for UI

Next we define a function to analyze the bias of a given article, a function to analyze the truthfulness of the given article, and a function to tie them together.  We found that joblib has limitations when it comes to functions that include other functions so we redefine our preprocessing functions here as well. 

In [17]:
# Text cleaning function
def clean_text(text):
    text = str(text).lower() #convert to lowercase
    text = re.sub(r'htt\S+', '', text) #remove urls
    text = re.sub(r'\d+', '', text) #remove numbers
    text = text.translate(str.maketrans('', '', string.punctuation)) #remove all other punctuation
    text = text.strip() # remove extra whitespace
    return text

In [18]:
# Initialize stopwords and lemmatizer
stop_words = set(stopwords.words('english'))
stop_words.add("reuters") #adds common stopwords
keep_words = {"not", "no", "never", "neither", "nor", "none", "but", "however", "us", "usa"} # words we want to keep
rem_words = {"via", "say", "says", "said", "one"} # extra words to add to stop_words
custom_stopw = stop_words - keep_words | rem_words

lemmatizer = WordNetLemmatizer()
exempt_words = {'usa', 'us', 'states'} # words to exempt from lemmatizer

In [19]:
# define function for tokenizing, lemmatization, and removal of stopwords
def preprocess_text(text):
    tokens = word_tokenize(str(text).lower())
    tokens = [t for t in tokens if t.isalpha() and t not in custom_stopw]
    tokens = [w if w in exempt_words else lemmatizer.lemmatize(w) for w in tokens]
    return " ".join(tokens)

#### Analyze Functions for UI

Here we define three functions to run our user interface.  Analyze_bias() runs the bias detection model.  Analyze_fake_news() runs the fake news model and anaylze_both() ties them together to simplify the interface code.  

In [21]:
def analyze_bias(user_first, news_text):

    clean_news = preprocess_text(news_text)

    X = bias_vectorizer.transform([clean_news])
    pred = bias_model.predict(X)[0]
    prob = bias_model.predict_proba(X).max()

    label = "left-leaning" if pred == 0 else "right-leaning"

    # --- NEW: get top contributing words ---
    feature_names = bias_vectorizer.get_feature_names_out()
    coefs = bias_model.coef_[0]

    indices = X.nonzero()[1]

    contributions = []
    for i in indices:
        value = X[0, i]
        contribution = value * coefs[i]
        contributions.append((feature_names[i], contribution))

    # sort by absolute contribution and take top 5
    top_words = sorted(contributions, key=lambda x: abs(x[1]), reverse=True)[:5]
    top_words = [word for word, _ in top_words]
    # --- end new section ---

    return f"Hi {user_first}, your article has a {label} political bias. I am {prob*100:.1f}% certain. Top words: {', '.join(top_words)}."

In [20]:
def analyze_fake_news(user_first, news_text):

    clean_news = preprocess_text(news_text)
    
    X = real_vectorizer.transform([clean_news])
    pred = real_model.predict(X)[0]
    prob = real_model.predict_proba(X).max()

    label = "Fake" if pred == 0 else "Real"

    """
    # --- NEW: get top contributing words ---
    feature_names = tdfit.get_feature_names_out()
    coefs = grid_best_estimator_3.coef_[0]

    indices = X.nonzero()[1]

    contributions = []
    for i in indices:
        value = X[0, i]
        contribution = value * coefs[i]
        contributions.append((feature_names[i], contribution))

    # sort by absolute contribution and take top 5
    top_words = sorted(contributions, key=lambda x: abs(x[1]), reverse=True)[:5]
    top_words = [word for word, _ in top_words]
    # --- end new section ---
"""
   # return f"Hi {user_first}, your article has a {label} political bias. I am {prob*100:.1f}% certain. Top words: {', '.join(top_words)}."
    return (f"Hi {user_first}, your article has been labelled as {label} news. I am {prob*100:.1f}% certain.")

In [22]:
def analyze_both(user_first, news_text):
    bias_result = analyze_bias(user_first, news_text)
    fake_result = analyze_fake_news(user_first, news_text)

    return bias_result + "\n\n" + fake_result

## Gradio User Interface

Next we code the gradio interface.  This pops up a browser window that asks you to enter your name and the text you want to analyze.  The app will then preprocess the text and run both the bias model and the fake news model to predict whether the text is real or fake and whether it is left or right leaning.  

*Note - You cannot cut and paste into the notebook version of the UI.  You must cut and paste the text into the browser pop-up of the user interface.*

In [23]:
with gr.Blocks(css="""
#header { text-align: center; color: darkblue; font-family: Arial; }
.primary-btn { background-color: #4CAF50; color: white; font-weight: bold; }

.result-text {
    background-color: white;
    padding: 12px;
    border-radius: 10px;
    border-left: 6px solid #b30000;
}

.result-text, .result-text * {
    color: #333333 !important;
    font-size: 18px;
}
""") as app:

    gr.Markdown("# Welcome to the News Detector", elem_id="header")

    with gr.Column(visible=True) as info_screen:

        user_first = gr.Textbox(label="Name", placeholder="Enter your first name")

        news = gr.Textbox(label="News Article Text", placeholder="Enter text here")

        submit_txt_btn = gr.Button("Submit Text for Analysis", elem_classes="primary-btn")

        result = gr.Textbox(label = "Results", elem_classes="result-text")

    submit_txt_btn.click(
        fn=analyze_both,
        inputs=[user_first, news],
        outputs=result
    )

app.launch(inbrowser=True, share=True)

/var/folders/wx/xm00n11n6xj5qds_xsdqlrvr0000gn/T/ipykernel_88220/1321461392.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css="""


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://65a213d7d373b2b5e5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
